# T5-base inverter on Llama-2-7B next-token log-probs (Dolly-15k)

Reproduction of *Language Model Inversion* (Morris et al.): T5-base learns to recover a prompt
from the target LM's next-token probability vector. The Llama-2-7B log-probs were precomputed
for every Dolly-15k instruction, so Llama-2 itself never runs here; only T5 is trained.

**Inputs**
- **Code:** cloned from GitHub (vec2text plus the patches for training from precomputed vectors).
- **Dataset:** `dolly15k-llama2-7b-logprobs`. It holds `llama2_dolly15k.npz` (instructions + 15011×32000 log-probs) and the Llama-2 tokenizer files (no weights).

**Settings**
- Accelerator: **one GPU T4**, with internet on.
- Set `SMOKE = True` for a ~60-step test run, and `SMOKE = False` for the full training.

In [ ]:
REPO_URL = "https://github.com/Yae-sine/Language-Model-Inversion.git"
REPO_BRANCH = "main"

SMOKE = True        # True: ~60-step test run; False: full training
NUM_EPOCHS = 40     # full run; one epoch = ~438 steps at batch 32

REPO_DIR = "/tmp/lmi"
V2T_DIR = f"{REPO_DIR}/vec2text"
NPY_DIR = "/tmp/llama2_dolly15k"          # extracted instructions.npy + vectors.npy
DATASET_DIR = "/tmp/dolly_llama2_inv"     # HF dataset built from them (~1.8 GB)
WORK = "/kaggle/working"                  # outputs (kept after the run)
OUT_DIR = f"{WORK}/saves/" + ("smoke" if SMOKE else "t5base_llama7b_dolly")
LOG_FILE = f"{WORK}/" + ("smoke" if SMOKE else "train") + ".log"

In [ ]:
import glob, json, os, shutil, subprocess, sys, zipfile


def sh(cmd, cwd=None, env=None, log=None):
    # Run a shell command, stream its output into the notebook, fail loudly on error.
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=env, executable="/bin/bash",
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    f = open(log, "a") if log else None
    for line in p.stdout:
        print(line, end="", flush=True)
        if f:
            f.write(line)
            f.flush()
    rc = p.wait()
    if f:
        f.close()
    if rc != 0:
        raise RuntimeError(f"command failed with exit code {rc}: {cmd}")


sh("nvidia-smi")
sh(f"rm -rf {REPO_DIR} && git clone --depth 1 -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}")
sh(f"git -C {REPO_DIR} log --oneline -1")

## Environment

`transformers` is pinned to 4.45.2. That is the last release that still accepts the
`evaluation_strategy` setting vec2text's `TrainingArguments` uses (removed in 4.46). It is also the
first release that works with `tokenizers` 0.20, which ships Python 3.13 builds; older `tokenizers`
fail to install on Kaggle's Python 3.13. A constraints file keeps Kaggle's own torch build.

In [ ]:
torch_ver = subprocess.check_output(
    [sys.executable, "-c", "import torch; print(torch.__version__)"], text=True).strip()
with open("/tmp/constraints.txt", "w") as f:
    f.write(f"torch=={torch_ver}\n")
print("keeping torch", torch_ver)

sh('pip install -q -c /tmp/constraints.txt "transformers==4.45.2" "tokenizers==0.20.3" "datasets>=2.20,<4" '
   '"accelerate==0.34.2" "sentence-transformers==3.0.1" evaluate sacrebleu rouge_score nltk '
   'sentencepiece einops tenacity openai')
sh(f"pip install -q -e {V2T_DIR} --no-deps")
sh("python -c \"import nltk; nltk.download('punkt', quiet=True); nltk.download('punkt_tab', quiet=True)\"")
sh("python -c \"import torch, transformers, datasets, vec2text; print('torch', torch.__version__, '| transformers', transformers.__version__, '| datasets', datasets.__version__, '| cuda', torch.cuda.is_available())\"")

## Inputs

The files are found by searching `/kaggle/input`, so this works whatever the dataset's mount path is,
and whether or not Kaggle unzipped the `.npz`.

The Llama-2 tokenizer and config are copied to `/kaggle/working/meta-llama/Llama-2-7b-hf`. Every
command then runs from `/kaggle/working`, so the model name `meta-llama/Llama-2-7b-hf` resolves to
that local folder, and the gated Hugging Face repo is never contacted.

In [ ]:
def find(name):
    return sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))


os.makedirs(NPY_DIR, exist_ok=True)
npz = [p for p in find("llama2_dolly15k.npz") if os.path.isfile(p) and zipfile.is_zipfile(p)]
if npz:
    print("extracting", npz[0])
    with zipfile.ZipFile(npz[0]) as z:
        for member in ("instructions.npy", "vectors.npy"):
            if not os.path.exists(f"{NPY_DIR}/{member}"):
                z.extract(member, NPY_DIR)
else:  # Kaggle already unzipped it
    for member in ("instructions.npy", "vectors.npy"):
        src = find(member)[0]
        print("linking", src)
        if not os.path.exists(f"{NPY_DIR}/{member}"):
            os.symlink(src, f"{NPY_DIR}/{member}")

tok_dir = os.path.dirname(find("tokenizer.model")[0])
llama_dir = f"{WORK}/meta-llama/Llama-2-7b-hf"
os.makedirs(llama_dir, exist_ok=True)
for name in ("config.json", "generation_config.json", "special_tokens_map.json",
             "tokenizer_config.json", "tokenizer.json", "tokenizer.model"):
    shutil.copy(f"{tok_dir}/{name}", llama_dir)

sh(f"ls -la {NPY_DIR} {llama_dir}")

## Build the training dataset

`build_dolly_llama2_dataset.py` does the following:
- tokenizes each instruction with the repo's own `tokenize_function`
- drops the instructions longer than 64 T5 tokens
- attaches each log-prob vector as the `frozen_embeddings` column
- splits the data 95/5 into train and validation (seed 42)

In [ ]:
if not os.path.exists(DATASET_DIR):
    sh(f"python {V2T_DIR}/experiment_scripts/logits/build_dolly_llama2_dataset.py "
       f"--data_dir {NPY_DIR} --out {DATASET_DIR}", cwd=WORK)
else:
    print("dataset already built:", DATASET_DIR)

## Train

These are the paper's settings: AdamW at lr 2e-4 with constant LR after warmup, and batch 32.

**Precision:** fp32, because the T4 has no bf16 and T5 tends to produce NaN losses in fp16.

**One GPU only:** the model updates a running "unigram" average inside `forward()`. Under the
Trainer's automatic DataParallel, those updates would be lost, so training uses `CUDA_VISIBLE_DEVICES=0`.

**Steps:** vec2text rescales `warmup/logging/eval/save_steps` by `32 / batch_size`, so at batch 32
they are plain optimizer steps.

**What the environment variables do:**
- `VEC2TEXT_TRAIN_DATASET_PATH` loads the dataset built above.
- `VEC2TEXT_EMBEDDER_NO_WEIGHTS` builds Llama-2 from its config only.
- `VEC2TEXT_SKIP_STANDARD_VAL` evaluates on the Dolly validation split only.
- `--mock_embedder 1` drops the (weightless) Llama-2 before training starts.

**Resuming:** to resume an interrupted run, attach the previous version's output as an input,
copy its last `checkpoint-*` folder into an empty `OUT_DIR`, and run again. vec2text resumes from it.

In [ ]:
train_args = {
    "experiment": "inversion_from_logits_emb",
    "model_name_or_path": "t5-base",
    "embedder_model_name": "meta-llama/Llama-2-7b-hf",
    "embedder_torch_dtype": "float16",
    "dataset_name": "dolly",
    "max_seq_length": 64,
    "use_frozen_embeddings_as_input": 1,
    "mock_embedder": 1,
    "per_device_train_batch_size": 32,
    "per_device_eval_batch_size": 32,
    "learning_rate": 2e-4,
    "lr_scheduler_type": "constant_with_warmup",
    "warmup_steps": 250,
    "num_train_epochs": NUM_EPOCHS,
    "eval_steps": 438,          # once per epoch
    "save_steps": 438,
    "logging_steps": 25,
    "save_total_limit": 2,      # plus the best checkpoint (by eval dolly_loss)
    "max_eval_samples": 500,
    "use_wandb": 0,
    "output_dir": OUT_DIR,
}
if SMOKE:
    train_args.update(max_steps=60, warmup_steps=10, eval_steps=30, save_steps=30,
                      logging_steps=5, max_eval_samples=32)

env = dict(os.environ,
           CUDA_VISIBLE_DEVICES="0",
           PYTHONUNBUFFERED="1",
           TOKENIZERS_PARALLELISM="false",
           VEC2TEXT_TRAIN_DATASET_PATH=DATASET_DIR,
           VEC2TEXT_EMBEDDER_NO_WEIGHTS="1",
           VEC2TEXT_SKIP_STANDARD_VAL="1")
cmd = f"python {V2T_DIR}/vec2text/run.py " + " ".join(f"--{k} {v}" for k, v in train_args.items())
sh(cmd, cwd=WORK, env=env, log=LOG_FILE)

## Results

In [ ]:
with open(f"{OUT_DIR}/trainer_state.json") as f:
    state = json.load(f)
print("best eval dolly_loss:", state.get("best_metric"), "at", state.get("best_model_checkpoint"))
keys = ["eval_dolly_loss", "eval_dolly_bleu_score", "eval_dolly_token_set_f1",
        "eval_dolly_exact_match", "eval_dolly_rouge_score"]
print("epoch  " + "  ".join(k.replace("eval_dolly_", "") for k in keys))
for e in state["log_history"]:
    if "eval_dolly_loss" in e:
        print(f"{e['epoch']:5.2f}  " + "  ".join(f"{e.get(k, float('nan')):.4f}" for k in keys))
train_end = [e for e in state["log_history"] if "train_runtime" in e]
if train_end:
    print("train runtime (s):", train_end[-1]["train_runtime"],
          "| steps/s:", train_end[-1]["train_steps_per_second"])

if SMOKE:  # keep the log, drop the ~GB of smoke checkpoints from the saved output
    shutil.rmtree(OUT_DIR)